# Exploration: why one churn threshold cannot serve every customer

The model is the easy part. This notebook is about the number that actually decides who gets
contacted, and why it has to differ per customer.

Real Telco data; the retention economics are declared assumptions, isolated in `config.py`.

In [1]:
import sys
sys.path.insert(0, "src")

import numpy as np
import pandas as pd

from churn.config import ECONOMICS
from churn.data import load
from churn.economics import break_even_probability, customer_value

frame = load()
print(f"{len(frame):,} customers, {frame['churned'].mean():.1%} churned")
print(f"monthly charges: R$ {frame['MonthlyCharges'].min():.2f} .. {frame['MonthlyCharges'].max():.2f}")

7,043 customers, 26.5% churned
monthly charges: R$ 18.25 .. 118.75


## 1. The data has one quiet trap

`TotalCharges` is stored as text and is blank for a handful of customers. Where the blanks
sit decides how to fill them.

In [2]:
raw = pd.read_csv("data/raw/telco_churn.csv")
blank = pd.to_numeric(raw["TotalCharges"], errors="coerce").isna()
print(f"blank TotalCharges: {blank.sum()} rows")
print(f"their tenure values: {sorted(raw.loc[blank, 'tenure'].unique())}")
print()
print("All of them have tenure 0: brand-new customers who have not been billed yet.")
print("Filling with the column mean would invent billing history and put tenure in")
print("contradiction with TotalCharges, which a tree model will happily exploit.")

blank TotalCharges: 11 rows
their tenure values: [np.int64(0)]

All of them have tenure 0: brand-new customers who have not been billed yet.
Filling with the column mean would invent billing history and put tenure in
contradiction with TotalCharges, which a tree model will happily exploit.


## 2. Customers are not equally worth saving

Value at risk is the contribution margin the business loses if they leave — not their
monthly bill.

In [3]:
value = customer_value(frame["MonthlyCharges"].to_numpy())
print(f"value at risk: R$ {value.min():.2f} .. R$ {value.max():.2f}  (ratio {value.max()/value.min():.1f}x)")
print(f"\nassumptions: {ECONOMICS.gross_margin:.0%} margin over "
      f"{ECONOMICS.expected_remaining_months} months, offer R$ {ECONOMICS.offer_cost:.0f}, "
      f"contact R$ {ECONOMICS.contact_cost:.0f}")

value at risk: R$ 120.45 .. R$ 783.75  (ratio 6.5x)

assumptions: 55% margin over 12 months, offer R$ 45, contact R$ 6


## 3. So the break-even threshold is not one number

Solving "expected value of contacting = 0" for the churn probability leaves value at risk in
the denominator:

```
break-even P(churn) = contact_cost / (acceptance_rate * (value_at_risk - offer_cost))
```

In [4]:
threshold = break_even_probability(value)
summary = pd.Series(threshold).describe(percentiles=[0.1, 0.5, 0.9])
print(summary[["min", "10%", "50%", "90%", "max"]].round(4).to_string())
print(f"\nspread between the 10th and 90th percentile: {summary['90%'] / summary['10%']:.1f}x")

min    0.0232
10%    0.0271
50%    0.0409
90%    0.1963
max    0.2272

spread between the 10th and 90th percentile: 7.2x


In [5]:
# What that looks like across the bill range
bands = pd.DataFrame({"monthly_charges": [20, 35, 50, 70, 90, 110]})
bands["value_at_risk"] = customer_value(bands["monthly_charges"].to_numpy()).round(2)
bands["break_even_churn_prob"] = break_even_probability(bands["value_at_risk"].to_numpy()).round(4)
print(bands.to_string(index=False))

 monthly_charges  value_at_risk  break_even_churn_prob
              20          132.0                 0.1970
              35          231.0                 0.0922
              50          330.0                 0.0602
              70          462.0                 0.0411
              90          594.0                 0.0312
             110          726.0                 0.0252


A customer on R$110 a month is worth contacting at a **3%** churn risk. One on R$20 a month
is not worth it until roughly **20%**.

A single global cutoff has to sit somewhere in between, and is therefore wrong in both
directions at once: it over-contacts cheap customers and under-contacts valuable ones.

## 4. What that costs, on held-out customers

In [6]:
import json
metrics = json.load(open("reports/metrics.json"))
rows = [
    {"rule": name, "contacted": s["contacted"], "saved": s["customers_saved"],
     "net_value": round(s["net_value"], 2)}
    for name, s in metrics["strategies"].items()
]
table = pd.DataFrame(rows).sort_values("net_value", ascending=False)
best = table["net_value"].max()
table["vs_best"] = ((table["net_value"] - best) / best).map(lambda v: f"{v:+.1%}")
print(table.to_string(index=False))

            rule  contacted  saved  net_value vs_best
  expected_value        975    135   48621.33   +0.0%
         best_f1        701    123   48237.33   -0.8%
    accuracy_0.5        450     92   38335.65  -21.2%
contact_everyone       1761    146   36950.31  -24.0%


## What this means for the decision

The 0.5 default — the one that falls out of `predict()` — leaves about **21%** of the
achievable value unclaimed, by contacting a third as many customers as it should.

Contacting everyone saves the most customers and destroys the most value: the offer cost
lands on everyone, including the people who were never leaving.

Honest caveat: against a threshold tuned for F1, the economic rule wins by well under 1% here.
The gap widens as customer value spreads out, and the result worth defending is the one
against 0.5.